# 10 - PINN 训练技巧实验

PINN 很依赖采样点、损失权重和网络容量。本 notebook 用一维 Poisson 方程做小实验：

- 改变配点数量
- 改变边界损失权重
- 比较最终最大误差

为了运行快，每组训练轮数较少。你可以把 `epochs` 调大看更稳定的结论。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=32, depth=3):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def exact_solution(x):
    return torch.sin(math.pi * x)

def residual(model, x):
    x.requires_grad_(True)
    u = model(x)
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    forcing = -(math.pi**2) * torch.sin(math.pi * x)
    return u_xx - forcing

In [ ]:
def run_experiment(n_f=128, bc_weight=50.0, epochs=2500, seed=0):
    torch.manual_seed(seed)
    model = MLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    xb = torch.tensor([[0.0], [1.0]], device=device)
    ub = torch.zeros_like(xb)
    history = []
    for _ in range(epochs):
        xf = torch.rand(n_f, 1, device=device)
        loss_phys = torch.mean(residual(model, xf)**2)
        loss_bc = torch.mean((model(xb) - ub)**2)
        loss = loss_phys + bc_weight * loss_bc
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
    x = torch.linspace(0, 1, 300, device=device).reshape(-1, 1)
    with torch.no_grad():
        err = torch.max(torch.abs(model(x) - exact_solution(x))).item()
    return err, history

configs = [
    {"n_f": 32, "bc_weight": 10.0},
    {"n_f": 128, "bc_weight": 10.0},
    {"n_f": 128, "bc_weight": 50.0},
    {"n_f": 512, "bc_weight": 50.0},
]

results = []
for i, cfg in enumerate(configs):
    err, hist = run_experiment(**cfg, seed=i)
    results.append((cfg, err, hist))
    print(cfg, "max_error=", f"{err:.3e}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
for cfg, err, hist in results:
    plt.semilogy(hist, label=f"n={cfg['n_f']}, w={cfg['bc_weight']}")
plt.legend(fontsize=8)
plt.title("Loss history")

plt.subplot(1, 2, 2)
labels = [f"n={cfg['n_f']}\nw={cfg['bc_weight']}" for cfg, _, _ in results]
errors = [err for _, err, _ in results]
plt.bar(labels, errors)
plt.yscale("log")
plt.title("Final max error")
plt.tight_layout()
plt.show()

经验上：配点太少会导致残差约束稀疏；边界权重太小会导致边界漂移；边界权重太大又可能压制 PDE 残差。真实项目中通常需要观察每一项损失，而不只看总损失。